# kdd_seq_06_sequential_gnn: Contrastive Session-Graph Neural Network

Implements a Sequence Graph Neural Network (SR-GNN style). It constructs an adjacency matrix dynamically per sequence, performs message passing via a Gated Graph Neural Network (GGNN), and applies soft attention readout. It relies entirely on standard PyTorch (no PyTorch Geometric required) ensuring it runs rapidly on a Colab T4 GPU.

In [4]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

WORK = "/content/drive/MyDrive/kdd_seq"
sys.path.insert(0, WORK)
import kdd_seq_common as C

# ------------------------------- CONFIG -------------------------------
METHOD = "gru_gnn"
MAX_LEN = 128
BATCH_SIZE = 512
EPOCHS = 15
LR = 3e-3
WEIGHT_DECAY = 1e-4
K_GRID = list(range(2, 9))
K_FINAL = 8
DOWNSTREAM_KS = [5, 8]
MIN_SHARE = 0.03
CV_SAMPLE, CV_SPLITS, CV_REPEATS = 60000, 3, 3
SEEDS_STAB, N_BOOT = 5, 8
SEED = 42
# ----------------------------------------------------------------------
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(f"{WORK}/results", exist_ok=True)

A = C.load_artifacts(WORK)
V = A["vocab"]["V"]; NAMES = A["vocab"]["names"]
F_tr, F_te, y_tr, y_te = A["F_train"], A["F_test"], A["y_train"], A["y_test"]
nev_tr, nev_te = A["base_train"]["n_events"].values, A["base_test"]["n_events"].values
print(f"Device: {DEVICE} | Train: {F_tr.shape[0]} | Test: {F_te.shape[0]} | V={V}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | Train: 120542 | Test: 80362 | V=19


In [5]:
# ==============================================================================
# Shared Sequence Packing, Spherical Clustering & Evaluation Helpers
# ==============================================================================
BOS = V          # 19
EOS = V + 1      # 20
PAD = V + 2      # 21
VOCAB_SIZE = V + 3  # 22

def pack_sequences(tokens, offsets, max_len=128):
    N = len(offsets) - 1
    lengths = np.diff(offsets)
    L_cap = int(min(max_len, lengths.max()))
    out = np.full((N, L_cap + 2), PAD, dtype=np.int64)
    out[:, 0] = BOS
    eff_lens = np.minimum(lengths, L_cap).astype(np.int64)
    for i in range(N):
        s, e = offsets[i], offsets[i + 1]
        L = e - s
        if L > L_cap:
            s = e - L_cap
            L = L_cap
        out[i, 1:1 + L] = tokens[s:e]
        out[i, 1 + L] = EOS
    return out, eff_lens, lengths.astype(np.int64)

class SphericalClusterer:
    def __init__(self, K, seed=0, n_init=10, temp=0.15):
        self.K = K; self.seed = seed; self.n_init = n_init; self.temp = temp
    def fit(self, Z):
        from sklearn.cluster import KMeans
        km = KMeans(self.K, n_init=self.n_init, random_state=self.seed).fit(Z)
        counts = np.bincount(km.labels_, minlength=self.K)
        order = np.argsort(-counts)
        self.centroids_ = km.cluster_centers_[order]
        norms = np.linalg.norm(self.centroids_, axis=1, keepdims=True)
        self.centroids_norm_ = self.centroids_ / np.maximum(norms, 1e-12)
        self.inertia_ = float(km.inertia_)
        return self
    def cosine_sim(self, Z):
        Zn = Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-12)
        return (Zn @ self.centroids_norm_.T).astype(np.float32)
    def predict_proba(self, Z):
        logits = self.cosine_sim(Z) / self.temp
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_l = np.exp(logits)
        return (exp_l / exp_l.sum(axis=1, keepdims=True)).astype(np.float32)
    def predict(self, Z):
        return self.cosine_sim(Z).argmax(axis=1)

def residualize_volume(Z_tr, Z_te, nev_tr, nev_te, len_tr, len_te):
    X_tr = np.column_stack([np.ones(len(nev_tr)), np.log1p(nev_tr), np.log1p(len_tr)])
    X_te = np.column_stack([np.ones(len(nev_te)), np.log1p(nev_te), np.log1p(len_te)])
    W, _, _, _ = np.linalg.lstsq(X_tr, Z_tr, rcond=None)
    R_tr, R_te = Z_tr - X_tr @ W, Z_te - X_te @ W
    R_tr /= np.maximum(np.linalg.norm(R_tr, axis=1, keepdims=True), 1e-12)
    R_te /= np.maximum(np.linalg.norm(R_te, axis=1, keepdims=True), 1e-12)
    return R_tr.astype(np.float32), R_te.astype(np.float32)

def describe_clusters_from_F(F, R_soft, V, names, alpha=0.1, cap=50, top=6, min_mass=0.005):
    K = R_soft.shape[1]
    n = np.asarray(F[:, V:].sum(1)).ravel()
    Fc = (sp.diags(np.minimum(1.0, cap / np.maximum(n, 1))) @ F).tocsr() if cap else F
    S = np.asarray(Fc.T.dot(R_soft)).T[:, V:].reshape(K, V, V)
    G = S.sum(0) + alpha
    Pg = G / G.sum(1, keepdims=True)
    out = {}
    for k in range(K):
        Pk = (S[k] + alpha) / (S[k] + alpha).sum(1, keepdims=True)
        mass = S[k] / np.maximum(S[k].sum(), 1e-12)
        lift = np.where(mass >= min_mass, Pk / Pg, 0)
        li = np.dstack(np.unravel_index(np.argsort(-lift, axis=None)[:top], lift.shape))[0]
        fi = np.dstack(np.unravel_index(np.argsort(-mass, axis=None)[:top], mass.shape))[0]
        out[k] = dict(share=float(R_soft[:, k].mean()),
                      characteristic=[f"{names[a]} -> {names[b]} (lift {lift[a,b]:.1f})" for a, b in li],
                      frequent=[f"{names[a]} -> {names[b]} ({mass[a,b]*100:.1f}%)" for a, b in fi])
    return out


X_tr, eff_tr, len_tr = pack_sequences(A["tok_train"], A["off_train"], MAX_LEN)
X_te, eff_te, len_te = pack_sequences(A["tok_test"],  A["off_test"],  MAX_LEN)
print("Packed train tensor:", X_tr.shape, "| Packed test tensor:", X_te.shape)


Packed train tensor: (120542, 130) | Packed test tensor: (80362, 130)


## 1. GNN Architecture & Unsupervised Pretraining
Trained strictly on augmented sequence views via NT-Xent contrastive loss.

In [6]:
def augment_batch_gpu(x, mask_prob=0.15, crop_min=0.6, unk_idx=18, bos_idx=BOS, eos_idx=EOS, pad_idx=PAD):
    """Fast GPU sequence augmentation (masking + cropping) to create contrastive views."""
    B, T = x.shape
    out = x.clone()
    device = x.device
    active = (out != pad_idx) & (out != bos_idx) & (out != eos_idx)
    lengths = active.sum(dim=1)

    rand_m = torch.rand((B, T), device=device)
    out[active & (rand_m < mask_prob)] = unk_idx

    crop_ratio = crop_min + (0.95 - crop_min) * torch.rand((B, 1), device=device)
    keep_len = torch.clamp((lengths.unsqueeze(1).float() * crop_ratio).long(), min=1)
    max_start = torch.clamp(lengths.unsqueeze(1) - keep_len, min=0)
    start_idx = (torch.rand((B, 1), device=device) * (max_start.float() + 1.0)).long() + 1
    end_idx = start_idx + keep_len

    pos = torch.arange(T, device=device).unsqueeze(0)
    drop_crop = active & (lengths.unsqueeze(1) >= 5) & ((pos < start_idx) | (pos >= end_idx))
    out[drop_crop] = pad_idx
    return out

class SessionGraphEncoder(nn.Module):
    """Session-based Graph Neural Network (SR-GNN style) Encoder.
    Builds an adjacency matrix dynamically per sequence, performs GGNN message passing,
    and applies soft-attention readout."""
    def __init__(self, vocab_size=VOCAB_SIZE, d_hid=64, pad_idx=PAD):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_idx = pad_idx
        self.emb = nn.Embedding(vocab_size, d_hid, padding_idx=pad_idx)
        self.W_in = nn.Linear(d_hid, d_hid, bias=False)
        self.W_out = nn.Linear(d_hid, d_hid, bias=False)
        self.gru = nn.GRUCell(2 * d_hid, d_hid)
        self.W_attn = nn.Linear(2 * d_hid, d_hid)
        self.q = nn.Linear(d_hid, 1, bias=False)

    def forward(self, x):
        B, T = x.shape
        device = x.device

        # 1. Build Adjacency matrices dynamically from sequence edges
        src, dst = x[:, :-1], x[:, 1:]
        src_oh = F.one_hot(src, num_classes=self.vocab_size).float()
        dst_oh = F.one_hot(dst, num_classes=self.vocab_size).float()

        A_out = torch.bmm(src_oh.transpose(1, 2), dst_oh) # (B, V, V) directed edges src -> dst
        A_in = A_out.transpose(1, 2)

        # Ignore connections involving PAD
        A_out[:, self.pad_idx, :] = 0; A_out[:, :, self.pad_idx] = 0
        A_in[:, self.pad_idx, :] = 0;  A_in[:, :, self.pad_idx] = 0

        A_out = F.normalize(A_out, p=1, dim=2)
        A_in = F.normalize(A_in, p=1, dim=2)

        # 2. Gated Graph Neural Network (GGNN) steps
        V_emb = self.emb.weight.unsqueeze(0).expand(B, -1, -1)
        for _ in range(2): # 2 hops of message passing
            msg_in = torch.bmm(A_in, self.W_in(V_emb))
            msg_out = torch.bmm(A_out, self.W_out(V_emb))
            msg = torch.cat([msg_in, msg_out], dim=-1)

            # CHANGED: Use .reshape() instead of .view() to handle non-contiguous memory
            msg_flat = msg.reshape(B * self.vocab_size, -1)
            V_flat = V_emb.reshape(B * self.vocab_size, -1)
            V_next = self.gru(msg_flat, V_flat)
            V_emb = V_next.reshape(B, self.vocab_size, -1)

        # 3. Soft-Attention Readout
        x_oh = F.one_hot(x, num_classes=self.vocab_size).float()
        seq_emb = torch.bmm(x_oh, V_emb) # (B, T, d)

        mask = (x != self.pad_idx).float().unsqueeze(-1)
        v_global = (seq_emb * mask).sum(1) / mask.sum(1).clamp(min=1.0)

        v_global_expand = v_global.unsqueeze(1).expand(-1, T, -1)
        alpha = self.q(torch.sigmoid(self.W_attn(torch.cat([seq_emb, v_global_expand], dim=-1))))
        alpha = alpha.masked_fill(mask == 0, -1e9)
        alpha = F.softmax(alpha, dim=1)

        z = (seq_emb * alpha).sum(1)
        return F.normalize(z, p=2, dim=-1)

class ContrastiveSequentialGNN(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_hid=64, d_proj=32, temp=0.1, pad_idx=PAD):
        super().__init__()
        self.temp = temp
        self.encoder = SessionGraphEncoder(vocab_size, d_hid, pad_idx)
        self.proj = nn.Sequential(nn.Linear(d_hid, d_hid), nn.GELU(), nn.Linear(d_hid, d_proj))

    def loss(self, x):
        x1 = augment_batch_gpu(x)
        x2 = augment_batch_gpu(x)
        h1, h2 = self.encoder(x1), self.encoder(x2)
        z1 = F.normalize(self.proj(h1), p=2, dim=-1)
        z2 = F.normalize(self.proj(h2), p=2, dim=-1)
        B = z1.size(0)
        z = torch.cat([z1, z2], dim=0)
        sim = torch.matmul(z, z.T) / self.temp
        sim.fill_diagonal_(-1e9)
        targets = torch.cat([torch.arange(B, 2 * B, device=z.device),
                             torch.arange(0, B, device=z.device)], dim=0)
        return F.cross_entropy(sim, targets)

    @torch.no_grad()
    def encode(self, x):
        return self.encoder(x)

MODEL_CLS = ContrastiveSequentialGNN

idx_tr, idx_va = train_test_split(np.arange(len(X_tr)), test_size=0.1, random_state=SEED)
train_loader = DataLoader(TensorDataset(torch.tensor(X_tr[idx_tr], dtype=torch.long)), batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(TensorDataset(torch.tensor(X_tr[idx_va], dtype=torch.long)), batch_size=BATCH_SIZE * 2, shuffle=False)

model = MODEL_CLS().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

best_val_loss, best_state = np.inf, None
for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_loss, n_tr = 0.0, 0
    for (bx,) in train_loader:
        bx = bx.to(DEVICE)
        opt.zero_grad()
        loss = model.loss(bx)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr_loss += loss.item() * len(bx); n_tr += len(bx)
    sched.step()
    model.eval()
    va_loss, n_va = 0.0, 0
    with torch.no_grad():
        for (bx,) in val_loader:
            bx = bx.to(DEVICE)
            va_loss += model.loss(bx).item() * len(bx); n_va += len(bx)
    tr_loss /= n_tr; va_loss /= n_va
    if va_loss < best_val_loss:
        best_val_loss = va_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if ep == 1 or ep % 3 == 0 or ep == EPOCHS:
        print(f"Epoch {ep:02d}/{EPOCHS} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | {time.time()-t0:.1f}s")

model.load_state_dict(best_state)
print(f"Best validation loss: {best_val_loss:.4f}")


Epoch 01/15 | train_loss=3.3260 | val_loss=3.4754 | 5.1s
Epoch 03/15 | train_loss=2.6740 | val_loss=3.2152 | 4.4s
Epoch 06/15 | train_loss=2.5565 | val_loss=3.1000 | 4.4s
Epoch 09/15 | train_loss=2.5062 | val_loss=3.0515 | 4.5s
Epoch 12/15 | train_loss=2.4794 | val_loss=3.0139 | 4.5s
Epoch 15/15 | train_loss=2.4684 | val_loss=2.9952 | 4.5s
Best validation loss: 2.9874


## 2. Embedding Extraction & Volume Residualization

In [7]:
@torch.no_grad()
def extract_embeddings(model, X_np, batch_size=1024):
    model.eval()
    zs = []
    for i in range(0, len(X_np), batch_size):
        bx = torch.tensor(X_np[i:i + batch_size], dtype=torch.long, device=DEVICE)
        zs.append(model.encode(bx).cpu().numpy())
    return np.vstack(zs).astype(np.float32)

Z_tr = extract_embeddings(model, X_tr)
Z_te = extract_embeddings(model, X_te)
Z_tr_orth, Z_te_orth = residualize_volume(Z_tr, Z_te, nev_tr, nev_te, len_tr, len_te)

pca16 = PCA(n_components=min(16, Z_tr.shape[1]), random_state=0).fit(Z_tr)
Z_tr_pca = pca16.transform(Z_tr).astype(np.float32)
Z_te_pca = pca16.transform(Z_te).astype(np.float32)
pca16_orth = PCA(n_components=min(16, Z_tr_orth.shape[1]), random_state=0).fit(Z_tr_orth)
Z_tr_orth_pca = pca16_orth.transform(Z_tr_orth).astype(np.float32)
Z_te_orth_pca = pca16_orth.transform(Z_te_orth).astype(np.float32)

print("Z_tr:", Z_tr.shape, "| PCA16 explained var: %.3f" % pca16.explained_variance_ratio_.sum())


Z_tr: (120542, 64) | PCA16 explained var: 0.926


## 3. Clustering Grid ($K = 2 \dots 8$)

In [8]:
rows, models = [], {}
for K in K_GRID:
    t0 = time.time()
    m = SphericalClusterer(K, seed=0, n_init=10).fit(Z_tr)
    models[K] = m
    lab_tr, lab_te = m.predict(Z_tr), m.predict(Z_te)
    P_tr = m.predict_proba(Z_tr)
    r = dict(K=K, inertia=m.inertia_, mean_max_posterior=float(P_tr.max(1).mean()))
    r.update(C.cluster_quality(A["eval_train"], lab_tr))
    nat = C.cluster_quality(Z_tr, lab_tr)
    r["native_silhouette"] = nat["silhouette"]
    r["native_davies_bouldin"] = nat["davies_bouldin"]
    r["native_calinski_harabasz"] = nat["calinski_harabasz"]
    r.update(C.volume_agreement(lab_tr, nev_tr))
    r["cramers_v_train"] = C.cramers_v(lab_tr, y_tr)
    r["cramers_v_test"]  = C.cramers_v(lab_te, y_te)
    rows.append(r)
    print(f"K={K} done in {time.time()-t0:.1f}s | shared Sil={r['silhouette']:.4f} | native Sil={r['native_silhouette']:.4f} | ARI_vol={r['ari_vs_volume']:.4f}")

grid = pd.DataFrame(rows).set_index("K")
grid.round(4).to_csv(f"{WORK}/results/{METHOD}_grid.csv")
display(grid.round(4))


K=2 done in 15.4s | shared Sil=0.1586 | native Sil=0.3529 | ARI_vol=0.6918
K=3 done in 15.2s | shared Sil=0.1780 | native Sil=0.3461 | ARI_vol=0.6718
K=4 done in 16.5s | shared Sil=0.1940 | native Sil=0.3275 | ARI_vol=0.5211
K=5 done in 17.8s | shared Sil=0.2215 | native Sil=0.3501 | ARI_vol=0.5343
K=6 done in 18.8s | shared Sil=0.2306 | native Sil=0.3574 | ARI_vol=0.5144
K=7 done in 19.9s | shared Sil=0.2366 | native Sil=0.3439 | ARI_vol=0.4818
K=8 done in 22.2s | shared Sil=0.2424 | native Sil=0.3464 | ARI_vol=0.4232


,inertia,mean_max_posterior,n_clusters,min_cluster_share,silhouette,davies_bouldin,calinski_harabasz,native_silhouette,native_davies_bouldin,native_calinski_harabasz,ari_vs_volume,nmi_vs_volume,eta2_logvolume,cramers_v_train,cramers_v_test
K,,,,,,,,,,,,,,,
2,61250.9609,0.9641,2,0.3825,0.1586,2.5657,2590.2332,0.3529,1.3217,10766.6523,0.6918,0.6543,0.6752,0.4148,0.4127
3,47870.8789,0.9060,3,0.3206,0.1780,3.1732,2141.5615,0.3461,1.3036,9660.0449,0.6718,0.6740,0.8424,0.4394,0.4331
4,43084.1367,0.8559,4,0.1628,0.1940,2.4340,2335.4121,0.3275,1.5833,7921.7266,0.5211,0.5852,0.8458,0.4269,0.4224
5,39234.4609,0.7800,5,0.1450,0.2215,3.4521,2007.8872,0.3501,1.6235,7025.9126,0.5343,0.6033,0.8807,0.4321,0.4265
6,36833.2500,0.7504,6,0.0920,0.2306,3.2205,1943.4761,0.3574,1.6478,6262.2495,0.5144,0.5728,0.8842,0.4279,0.4212
7,35043.8906,0.6924,7,0.0712,0.2366,3.1539,1755.1912,0.3439,1.7469,5660.0732,0.4818,0.5644,0.8940,0.4320,0.4269
8,33411.6289,0.6755,8,0.0456,0.2424,3.0109,1725.6603,0.3464,1.7313,5229.5420,0.4232,0.5456,0.8942,0.4286,0.4221


## 4. Cluster Profiles & Behavioural Interpretation

In [9]:
for K_view in DOWNSTREAM_KS:
    print(f"\n=================== {METHOD.upper()} PROFILE (K={K_view}) ===================")
    mk = models[K_view]
    l_tr = mk.predict(Z_tr)
    print(C.cluster_profile(l_tr, y_tr, nev_tr))
    desc = describe_clusters_from_F(F_tr, mk.predict_proba(Z_tr), V, NAMES)
    json.dump(desc, open(f"{WORK}/results/{METHOD}_cluster_descriptions_K{K_view}.json", "w"), indent=1)
    for k, d in desc.items():
        print(f"\n--- Cluster {k} (soft share {d['share']:.3f}) ---")
        print("  characteristic:", "; ".join(d["characteristic"][:4]))
        print("  frequent:      ", "; ".join(d["frequent"][:4]))



=================== GRU_GNN PROFILE (K=5) ===================
    size  share  dropout  mean_events  median_events
c                                                   
0  34225  0.284    0.889       18.050           16.0
1  26285  0.218    0.957        1.489            1.0
2  23376  0.194    0.552      103.977           78.0
3  19177  0.159    0.566      146.332           96.0
4  17479  0.145    0.930        6.342            6.0

--- Cluster 0 (soft share 0.256) ---
  characteristic: browser_video_video -> browser_page_close_unknown (lift 2.0); browser_video_unknown -> browser_page_close_unknown (lift 1.9); server_access_sequential -> server_navigate_unknown (lift 1.8); server_access_unknown -> server_access_sequential (lift 1.7)
  frequent:       server_access_sequential -> browser_page_close_unknown (11.7%); server_navigate_unknown -> server_access_unknown (10.2%); browser_page_close_unknown -> browser_video_video (9.1%); server_access_unknown -> server_access_sequential (6.5%)

---

## 5. Stability Analysis

In [10]:
stab_by_k = {}
for K_eval in DOWNSTREAM_KS:
    mk = models[K_eval]
    lab_tr_k, lab_te_k = mk.predict(Z_tr), mk.predict(Z_te)
    seed_labels = [SphericalClusterer(K_eval, seed=100 + s, n_init=5).fit(Z_tr).predict(Z_tr) for s in range(SEEDS_STAB)]
    st = C.seed_stability(seed_labels)
    st.update(C.bootstrap_stability(
        fit=lambda idx, seed: SphericalClusterer(K_eval, seed=seed, n_init=5).fit(Z_tr[idx]),
        predict=lambda cm, rows: cm.predict(Z_tr[rows]),
        n_rows=Z_tr.shape[0], ref_labels=lab_tr_k, n_boot=N_BOOT))
    st.update(C.transfer_consistency(lab_tr_k, y_tr, lab_te_k, y_te, K_eval))
    stab_by_k[K_eval] = {k: round(v, 4) for k, v in st.items()}
    print(f"\n=== Stability ({METHOD}, K={K_eval}) ===")
    print(pd.Series(stab_by_k[K_eval]).to_string())



=== Stability (gru_gnn, K=5) ===
seed_ari_mean                 0.8985
seed_ari_min                  0.8305
boot_ari_mean                 0.9383
boot_ari_std                  0.0773
share_L1_train_vs_test        0.0029
dropout_corr_train_vs_test    1.0000
cramers_v_train               0.4321
cramers_v_test                0.4265

=== Stability (gru_gnn, K=8) ===
seed_ari_mean                 0.8444
seed_ari_min                  0.7413
boot_ari_mean                 0.8998
boot_ari_std                  0.1238
share_L1_train_vs_test        0.0087
dropout_corr_train_vs_test    0.9998
cramers_v_train               0.4286
cramers_v_test                0.4221


## 6. Downstream Dropout Prediction

In [11]:
def make_variants(model_k, K, Z_a, Z_pca_a, Z_orth_pca_a, seed=0):
    P = model_k.predict_proba(Z_a)
    lab = P.argmax(1)
    cs = model_k.cosine_sim(Z_a)
    rng = np.random.default_rng(seed)
    return {"hard": C.onehot(lab, K), "soft": P.astype("float32"),
            "hard+soft": np.hstack([C.onehot(lab, K), P]).astype("float32"),
            "cos_sim": cs.astype("float32"), "emb_pca16": Z_pca_a.astype("float32"),
            "emb_vol_orth": Z_orth_pca_a.astype("float32"),
            "null_random": C.onehot(rng.integers(0, K, Z_a.shape[0]), K)}

cv_all, test_all = [], []
for K in DOWNSTREAM_KS:
    mk = models[K]
    var_tr = make_variants(mk, K, Z_tr, Z_tr_pca, Z_tr_orth_pca, seed=0)
    var_te = make_variants(mk, K, Z_te, Z_te_pca, Z_te_orth_pca, seed=1)
    summ, wide = C.downstream_cv(A["base_train"], y_tr, var_tr, CV_SAMPLE, CV_SPLITS, CV_REPEATS)
    summ.insert(0, "K", K); summ.insert(0, "method", METHOD); cv_all.append(summ)
    print(f"\n=== {METHOD} Train CV, K={K} ==="); display(summ)
    ft = C.final_test_eval(A["base_train"], y_tr, A["base_test"], y_te, var_tr, var_te)
    ft.insert(0, "K", K); ft.insert(0, "method", METHOD); test_all.append(ft)
    print(f"=== {METHOD} Real TEST Set, K={K} ==="); display(ft)

pd.concat(cv_all).to_csv(f"{WORK}/results/{METHOD}_downstream_cv.csv")
pd.concat(test_all).to_csv(f"{WORK}/results/{METHOD}_downstream_test.csv")



=== gru_gnn Train CV, K=5 ===


,method,K,auc_mean,auc_std,delta_vs_base,delta_std,folds_better
variant,,,,,,,
baseline,gru_gnn,5,0.85318,0.00205,0.00000,0.00000,0
hard,gru_gnn,5,0.85331,0.00222,0.00013,0.00035,7
soft,gru_gnn,5,0.85309,0.00188,-0.00009,0.00068,3
hard+soft,gru_gnn,5,0.85312,0.00208,-0.00006,0.00096,4
cos_sim,gru_gnn,5,0.85315,0.00182,-0.00003,0.00071,6
emb_pca16,gru_gnn,5,0.85312,0.00193,-0.00006,0.00077,4
emb_vol_orth,gru_gnn,5,0.85318,0.00247,-0.00001,0.00066,4
null_random,gru_gnn,5,0.85312,0.00221,-0.00007,0.00038,3


=== gru_gnn Real TEST Set, K=5 ===


method  K      auc                ap                f1  \
                             mean      std     mean      std     mean   
variant                                                                 
baseline      gru_gnn  5  0.85305  0.00006  0.94705  0.00008  0.91002   
hard          gru_gnn  5  0.85315  0.00021  0.94716  0.00018  0.91022   
soft          gru_gnn  5  0.85295  0.00017  0.94695  0.00012  0.91010   
hard+soft     gru_gnn  5  0.85294  0.00018  0.94694  0.00020  0.90987   
cos_sim       gru_gnn  5  0.85317  0.00019  0.94705  0.00018  0.91007   
emb_pca16     gru_gnn  5  0.85361  0.00012  0.94714  0.00008  0.91008   
emb_vol_orth  gru_gnn  5  0.85325  0.00021  0.94701  0.00012  0.91009   
null_random   gru_gnn  5  0.85309  0.00005  0.94706  0.00012  0.90989   

                      auc_delta_vs_base  
                  std                    
variant                                  
baseline      0.00028           0.00000  
hard          0.00052           0.00010  
soft          0.00034          -0.00010  
hard+soft     0.00037          -0.00011  
cos_sim       0.00040           0.00012  
emb_pca16     0.00034           0.00056  
emb_vol_orth  0.00018           0.00020  
null_random   0.00020           0.00004


=== gru_gnn Train CV, K=8 ===


,method,K,auc_mean,auc_std,delta_vs_base,delta_std,folds_better
variant,,,,,,,
baseline,gru_gnn,8,0.85318,0.00205,0.00000,0.00000,0
hard,gru_gnn,8,0.85322,0.00210,0.00004,0.00057,5
soft,gru_gnn,8,0.85287,0.00236,-0.00032,0.00086,5
hard+soft,gru_gnn,8,0.85303,0.00212,-0.00015,0.00075,4
cos_sim,gru_gnn,8,0.85324,0.00227,0.00006,0.00083,5
emb_pca16,gru_gnn,8,0.85312,0.00193,-0.00006,0.00077,4
emb_vol_orth,gru_gnn,8,0.85318,0.00247,-0.00001,0.00066,4
null_random,gru_gnn,8,0.85319,0.00190,0.00001,0.00051,5


=== gru_gnn Real TEST Set, K=8 ===


method  K      auc                ap                f1  \
                             mean      std     mean      std     mean   
variant                                                                 
baseline      gru_gnn  8  0.85305  0.00006  0.94705  0.00008  0.91002   
hard          gru_gnn  8  0.85322  0.00015  0.94711  0.00014  0.91015   
soft          gru_gnn  8  0.85311  0.00010  0.94703  0.00014  0.91018   
hard+soft     gru_gnn  8  0.85303  0.00013  0.94699  0.00016  0.90993   
cos_sim       gru_gnn  8  0.85324  0.00005  0.94708  0.00008  0.90985   
emb_pca16     gru_gnn  8  0.85361  0.00012  0.94714  0.00008  0.91008   
emb_vol_orth  gru_gnn  8  0.85325  0.00021  0.94701  0.00012  0.91009   
null_random   gru_gnn  8  0.85301  0.00006  0.94700  0.00001  0.90986   

                      auc_delta_vs_base  
                  std                    
variant                                  
baseline      0.00028           0.00000  
hard          0.00002           0.00017  
soft          0.00019           0.00006  
hard+soft     0.00024          -0.00002  
cos_sim       0.00022           0.00019  
emb_pca16     0.00034           0.00056  
emb_vol_orth  0.00018           0.00020  
null_random   0.00060          -0.00004

## 7. Save Artifacts & Summary JSON

In [12]:
np.save(f"{WORK}/results/feat_{METHOD}_train_emb.npy", Z_tr)
np.save(f"{WORK}/results/feat_{METHOD}_test_emb.npy",  Z_te)
np.save(f"{WORK}/results/feat_{METHOD}_train_emb_orth.npy", Z_tr_orth)
np.save(f"{WORK}/results/feat_{METHOD}_test_emb_orth.npy",  Z_te_orth)
torch.save(model.state_dict(), f"{WORK}/results/{METHOD}_weights.pt")

for K_save in DOWNSTREAM_KS:
    mk = models[K_save]
    for s, ZZ in (("train", Z_tr), ("test", Z_te)):
        P = mk.predict_proba(ZZ)
        np.save(f"{WORK}/results/feat_{METHOD}_K{K_save}_{s}_soft.npy", P.astype("float32"))
        np.save(f"{WORK}/results/feat_{METHOD}_K{K_save}_{s}_cossim.npy", mk.cosine_sim(ZZ).astype("float32"))
        pd.DataFrame(P, index=A[f"ids_{s}"], columns=[f"p{k}" for k in range(K_save)]).assign(cluster=P.argmax(1)) \
          .to_csv(f"{WORK}/results/{METHOD}_labels_K{K_save}_{s}.csv", index_label="enrollment_id")

summary = dict(method=METHOD, K=K_FINAL, max_len=MAX_LEN, val_loss=float(best_val_loss))
summary.update({k: float(v) for k, v in grid.loc[K_FINAL].items()})
summary.update(stab_by_k[K_FINAL])
summary["K5_summary"] = {**{k: float(v) for k, v in grid.loc[5].items()}, **stab_by_k[5]}
json.dump(summary, open(f"{WORK}/results/summary_{METHOD}.json", "w"), indent=1)
print(json.dumps(summary, indent=1))


{
 "method": "gru_gnn",
 "K": 8,
 "max_len": 128,
 "val_loss": 2.9874376849186763,
 "inertia": 33411.62890625,
 "mean_max_posterior": 0.6754711866378784,
 "n_clusters": 8.0,
 "min_cluster_share": 0.045610658525659105,
 "silhouette": 0.24244673550128937,
 "davies_bouldin": 3.01086374745111,
 "calinski_harabasz": 1725.6602783203125,
 "native_silhouette": 0.34644079208374023,
 "native_davies_bouldin": 1.731283683747824,
 "native_calinski_harabasz": 5229.5419921875,
 "ari_vs_volume": 0.4231521143472991,
 "nmi_vs_volume": 0.5455895709445747,
 "eta2_logvolume": 0.8941887400767028,
 "cramers_v_train": 0.4286,
 "cramers_v_test": 0.4221,
 "seed_ari_mean": 0.8444,
 "seed_ari_min": 0.7413,
 "boot_ari_mean": 0.8998,
 "boot_ari_std": 0.1238,
 "share_L1_train_vs_test": 0.0087,
 "dropout_corr_train_vs_test": 0.9998,
 "K5_summary": {
  "inertia": 39234.4609375,
  "mean_max_posterior": 0.7799934148788452,
  "n_clusters": 5.0,
  "min_cluster_share": 0.14500340130410977,
  "silhouette": 0.221491068601608